In [1]:
import numpy as np
# tensorflow: 이번 노트북의 주인공
# - tf.keras.Sequential : 모델을 쌓아 만드는 고수준 API
# - tf.keras.layers.Dense : H = a*X_norm + b를 계산하는 선형 부품 (PyTorch의 Linear에 대응)
# - tf.keras.losses.BinaryCrossentropy(from_logits=True) : H를 받아 sigmoid+BCE를 내부 처리 (BCEWithLogitsLoss에 대응)
# - tf.keras.optimizers.SGD : 파라미터 업데이트 (PyTorch의 SGD optimizer에 대응)
import tensorflow as tf
import pandas as pd

In [2]:
# - np.random.seed(42) : numpy 쪽 난수 고정
# - tf.random.set_seed(42) : TensorFlow 쪽 난수 고정 (Dense layer 초기 weight에 영향)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow 버전:", tf.__version__)
print("NumPy 버전:", np.__version__)
print("Pandas 버전:", pd.__version__)

TensorFlow 버전: 2.10.0
NumPy 버전: 1.26.4
Pandas 버전: 2.3.3


In [3]:
# 여기서는 H값을 직접 만든 뒤 sigmoid를 적용해 보기 위해 NumPy 배열을 사용합니다.
# 아직 모델을 학습하는 단계가 아니므로 TensorFlow Tensor로 만들 필요는 없습니다.
# shape를 (5, 1)로 만든 이유:
# 데이터가 5개 있음
# 각데이터는 H 값 1개를 가짐
# 따라서 모양은 (데이터개수, 1) = (5, 1)
H_example = np.array([[-3.0], [-1.0], [0.0], [1.0], [3.0]])

H_example

array([[-3.],
       [-1.],
       [ 0.],
       [ 1.],
       [ 3.]])

In [4]:
# [NumPy 배열 -> tf.sigmoid -> TensorFlow Tensor]
# H_example은 NumPy배열 입니다.
# tf.sigmoid()는 NumPy 배열도 입력으로 받을 수 있음
# tf.sigmoid(H_example)을 적용하면 TensorFlow가 이 값을 받아 계산하고,
# 그 결과인 z_example은 TensorFlow Tensor로 반환됩니다.
z_example = tf.sigmoid(H_example)

z_example

<tf.Tensor: shape=(5, 1), dtype=float64, numpy=
array([[0.04742587],
       [0.26894142],
       [0.5       ],
       [0.73105858],
       [0.95257413]])>

In [5]:
# H와 z를 표로 나란히 놓고 비교합니다. (pandas DataFrame 사용)
#
# [pd.DataFrame 이란?]
#   pd.DataFrame은 값을 '표(table)' 형태로 보여 주기 위한 도구입니다.
#   모델 학습에 꼭 필요한 코드는 아니지만, H와 z를 나란히 비교하면
#   'H는 확률이 아니고, z가 확률이다'라는 점을 눈으로 확인하기 쉽습니다.
#
# [reshape(-1) / .numpy() 가 왜 필요한가?]
#   H_example, z_example 은 shape이 (5, 1)인 2차원(5행 1열) 데이터입니다.
#   DataFrame의 한 '열(column)'로 넣으려면 [값1, 값2, 값3, ...] 형태의 1차원 배열이 편합니다.
#     - .reshape(-1)  : (5, 1) 같은 2차원 모양을 (5,) 1차원으로 '펴 줍니다'. (-1은 길이를 알아서 맞추라는 뜻)
#     - .numpy()      : TensorFlow Tensor를 NumPy 배열로 바꿉니다. (Tensor에만 필요)
sigmoid_result_df = pd.DataFrame({
    # H_example은 NumPy 배열이므로 .numpy()가 필요 없습니다.
    # reshape(-1)은 (5, 1) 모양을 DataFrame 열에 넣기 쉬운 1차원 형태로 펴는 역할입니다.
    "H": H_example.reshape(-1),                      # sigmoid 이전의 선형 출력값 (음수/0/양수 모두 가능)

    # z_example은 TensorFlow Tensor이므로 .numpy()로 NumPy 배열로 꺼냅니다.
    "z = sigmoid(H)": z_example.numpy().reshape(-1)  # sigmoid를 통과한 예측 확률 (항상 0~1 사이)
})

# sigmoid 함수는 H를 0과 1 사이의 값으로 바꿉니다.
# 따라서 sigmoid(H)의 결과인 z를 예측 확률로 해석합니다.
#   정리:  H = sigmoid 이전의 선형 출력값
#          z = sigmoid(H) = 예측 확률
sigmoid_result_df

,H,z = sigmoid(H)
0,-3.0,0.047426
1,-1.0,0.268941
2,0.0,0.500000
3,1.0,0.731059
4,3.0,0.952574


In [6]:
# 이진 분류에서는 예측 확률 z를 기준으로 최종 클래스를 결정합니다.
# 이번 입문 예제에서는 기본 기준값으로 0.5를 사용합니다.
#   z >= 0.5 이면 1
#   z <  0.5 이면 0
#
# [tf.cast(z_example >= 0.5, tf.int32) 을 초보자용으로 풀어 보면]
#   1) z_example >= 0.5 -> 각 원소가 조건을 만족하는지에 따라 True 또는 False 값을 만듭니다.
#   2) tf.cast(..., tf.int32) -> True를 1로, False를 0으로 '형 변환(cast)'합니다.
#   즉, 확률 z를 최종 분류값 0 또는 1로 바꾸는 코드입니다.
prediction_example = tf.cast(z_example >= 0.5, tf.int32)

# 앞에서 만든 표(sigmoid_result_df)에 prediction 열을 추가해 H, z, 최종 분류를 한눈에 봅니다.
# (여기서도 .numpy().reshape(-1)로 Tensor를 1차원 NumPy 배열로 펴서 표의 한 열로 넣습니다.)
sigmoid_result_df["prediction"] = prediction_example.numpy().reshape(-1)

sigmoid_result_df

,H,z = sigmoid(H),prediction
0,-3.0,0.047426,0
1,-1.0,0.268941,0
2,0.0,0.500000,1
3,1.0,0.731059,1
4,3.0,0.952574,1


In [7]:
# tf.keras.Input(shape=(1,)) : 입력이 '데이터 한 개당 특성 1개'라는 것을 알려 줍니다.
#                              (데이터 개수가 아니라 '특성 개수'라는 점에 주의!)
# tf.keras.layers.Dense(1)   : H = a·X_norm + b 를 계산하는 선형 층입니다.
#   ★ activation을 넣지 않습니다! 따라서 출력은 확률 z가 아니라 H(선형 계산값)입니다.
#   (sigmoid는 뒤에서 BinaryCrossentropy(from_logits=True)가 내부 처리합니다.)

# 모델을 만들기 직전에 seed를 한 번 더 고정합니다.
#   주의: tf.constant나 tf.sigmoid는 난수를 사용하는 연산이 아닙니다.
#         따라서 위 '1) sigmoid 복습' 준비 실습 자체가 Dense 초기 weight를 바꾸는 원인은 아닙니다.
#   그렇다면 왜 다시 고정할까요?
#     - 교육용 노트북에서는 모델 생성 직전에 seed를 다시 고정해 두면
#       '이제부터 모델을 만들겠다'는 초기화 시점을 명확히 표시할 수 있고,
#       실행 결과를 항상 같은 초기 weight에서 비교하기 쉽습니다. (재현성 표시)
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(1)
])

# 모델 구조를 요약해서 봅니다.
#   아래 출력에서 Param # 가 2로 보이는 이유는
#   Dense(1)이 weight a 1개 + bias b 1개 = 총 2개의 파라미터를 가지기 때문입니다.
#   즉, 이 모델이 학습으로 바꾸는 값은 a와 b 두 개뿐입니다.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________


In [8]:
# X: 입력값(사람의 키 cm). dtype=np.float32 로 실수 형식, reshape(-1, 1)로 (n, 1) 형태.
#    Dense(1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 (n, 1) 형태여야 합니다.
#
# [reshape(-1, 1)을 초보자용으로 더 자세히]
#   Dense(1)은 입력을 '표(table)' 형태로 받는다고 생각하면 쉽습니다.
#     - 행(row)    = 데이터 개수 (사람 수). 여기서는 키가 4개이므로 4행.
#     - 열(column) = 입력 특성 개수. 이번 예제는 '키' 하나만 쓰므로 특성은 1개 -> 1열.
#   따라서 X의 shape은 (데이터 개수, 1) = (4, 1) 이 되어야 합니다.
#   reshape(-1, 1) 에서:
#     -1 : 행(데이터 개수)은 NumPy가 알아서 계산하라는 뜻. (여기서는 4로 자동 결정됨)
#      1 : 각 데이터가 입력 특성 1개를 가진다는 뜻.
X = np.array([160, 170, 180, 190], dtype=np.float32).reshape(-1, 1)

# y: 정답값(0=농구선수 아님, 1=농구선수). 마찬가지로 (n, 1) 형태로 맞춥니다.
y = np.array([0, 0, 1, 1], dtype=np.float32).reshape(-1, 1)

print("입력값 X:\n", X)
print("정답값 y:\n", y)
print("X shape:", X.shape)    # (4, 1) 이어야 합니다.
print("y shape:", y.shape)    # (4, 1) 이어야 합니다.

입력값 X:
 [[160.]
 [170.]
 [180.]
 [190.]]
정답값 y:
 [[0.]
 [0.]
 [1.]
 [1.]]
X shape: (4, 1)
y shape: (4, 1)


In [9]:
# 평균(mean)과 표준편차(std)를 계산합니다.
# 이 X_mean, X_std 는 나중에 '새 입력값 예측'에서도 똑같이 다시 씁니다. (새로 계산하면 안 됩니다.)
X_mean = X.mean()
X_std = X.std()

# 정규화 공식: (원본값 - 평균) / 표준편차
# 주의: 실제 학습에는 원래 키 X 가 아니라, 정규화된 입력값 X_norm을 사용합니다.
X_norm = (X - X_mean) / X_std

print("입력값 평균 X_mean:", X_mean)
print("입력값 표준편차 X_std:", X_std)
print("정규화된 입력값 X_norm:\n", X_norm)
print("X_norm shape:", X_norm.shape)  # (4, 1) 이어야 합니다.

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.18034
정규화된 입력값 X_norm:
 [[-1.3416408]
 [-0.4472136]
 [ 0.4472136]
 [ 1.3416408]]
X_norm shape: (4, 1)


In [10]:
# model(X_norm) : 이번 구조에서는 H(선형 계산값)를 반환합니다.
H_before = model(X_norm)

# 예측 확률 z를 보려면 H에 sigmoid를 직접 적용해야 합니다.
z_before = tf.sigmoid(H_before)

print(H_before.numpy()[:5])
print("")
print(z_before.numpy()[:5])

[[-0.7002101 ]
 [-0.23340335]
 [ 0.23340335]
 [ 0.7002101 ]]

[[0.33176565]
 [0.44191262]
 [0.55808735]
 [0.66823435]]


In [11]:
learning_rate = 0.1
epochs = 1000
# model.compile : 학습 방법을 모델에 지정합니다.
# optimizer = SGD(...) -> PyTorch의 SGD optimizer 에 대응
# loss = BinaryCrossentropy(from_logits=True) -> PyTorch의 BCEWithLogitsLoss()에 대응
#       ★from_logits=True : model이 확률 z가 아니라 H를 출력한다는 뜻.
#           손실 함수가 내부에서 sigmoid와 BCE를 함께 처리합니다.
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True)
)

print(learning_rate)
print(epochs)
print("compile 완료 : optimizer=SGD, loss=BinaryCrossentropy(from_logits=True)")

0.1
1000
compile 완료 : optimizer=SGD, loss=BinaryCrossentropy(from_logits=True)


In [12]:
# model.fit(X_norm, y, ...):
#   - 입력으로 반드시 X_norm(정규화된 값)을 사용합니다.
#   - epochs 만큼 반복하며, 내부에서 자동으로
#   1. H계산 2. Cost계산 3. gradient계산 4. weight/bias 업데이트를 수행합니다.
#   - verbose=0 : 진행 막대를 숨깁니다. (학습 기록은 history에 저장됩니다.)
#
# [batch_size는 왜 안 적었나요?]
#   여기서는 batch_size를 따로 지정하지 않았습니다. Keras의 기본 batch_size는 보통 32입니다.
#   이번 예제 데이터는 4개뿐이므로, 한 epoch에서 4개 데이터가 사실상 한 번에 처리됩니다.
#   그래서 이전 PyTorch 노트북에서 전체 데이터의 mean_cost를 계산하던 흐름과 비교하기 쉽습니다.
history = model.fit(
    X_norm,
    y,
    epochs=epochs,
    verbose=0
)

print("학습 완료!")

# history.history는 학습 과정에서 기록된 값들을 dictionary(딕셔너리) 형태로 저장합니다.
# 지금은 compile에서 loss만 지정했기 때문에 주로 "loss" 항목이 들어 있습니다.
# (나중에 metrics=["accuracy"] 같은 것을 추가하면 "accuracy" 항목도 함께 기록될 수 있지만,
#   이번 노트북에서는 Cost 흐름에 집중하기 위해 metrics는 추가하지 않습니다.)
print("history에 기록된 항목:", list(history.history.keys()))

학습 완료!
history에 기록된 항목: ['loss']


In [13]:
# history.history["loss"] : Keras 내부 키 이름은 "loss" 이지만,
#                           그 값의 의미는 각 epoch의 '평균 Cost' = mean_cost 입니다.
#   - history              : model.fit()이 남긴 '학습 기록'입니다.
#   - history.history["loss"] : 각 epoch마다 계산된 평균 Cost가 순서대로 들어 있는 리스트입니다.
#   (Keras API 이름은 loss이지만, 이 교재에서는 이 값을 mean_cost, 즉 평균 Cost로 이해합니다.)
#
# 일정 epoch마다 평균 Cost를 출력해, 학습이 진행되며 줄어드는 모습을 확인합니다.
#
# [for 문과 enumerate 초보자용 설명]
#   - enumerate(리스트) : 리스트에서 '순서 번호'와 '값'을 함께 꺼내 주는 기능입니다.
#   - epoch_index      : 몇 번째 epoch인지 나타내는 순서 번호입니다. (0, 1, 2, ... 로 시작)
#   - mean_cost        : 그 epoch에서의 평균 Cost 값입니다.
#   - if 조건          : 100 epoch마다 한 번씩(그리고 마지막 epoch에서) 출력해, 화면이 너무 길어지지 않게 합니다.
for epoch_index, mean_cost in enumerate(history.history["loss"]):
    if epoch_index % 100 == 0 or epoch_index == epochs - 1:
        print(f"epoch={epoch_index}, mean_cost={mean_cost:.6f}")

# 마지막 epoch의 최종 평균 Cost
#   history.history["loss"][-1] : 리스트의 맨 마지막 값 = 학습이 끝난 시점의 평균 Cost(mean_cost)
final_mean_cost = history.history["loss"][-1]
print(f"\n최종 mean_cost(평균 Cost): {final_mean_cost:.6f}")

epoch=0, mean_cost=0.493178
epoch=100, mean_cost=0.186958
epoch=200, mean_cost=0.129461
epoch=300, mean_cost=0.101749
epoch=400, mean_cost=0.084570
epoch=500, mean_cost=0.072615
epoch=600, mean_cost=0.063721
epoch=700, mean_cost=0.056808
epoch=800, mean_cost=0.051262
epoch=900, mean_cost=0.046707
epoch=999, mean_cost=0.042930

최종 mean_cost(평균 Cost): 0.042930


In [14]:
# model.layers[-1] : 모델의 마지막 층 = Dense(1) 층. (Dense layer 안에 학습된 weight와 bias가 들어 있습니다.)
#   (Input 층은 가중치가 없으므로, 실제 weight/bias는 Dense 층에서 가져옵니다.)
# get_weights() : [weight 배열, bias 배열] 형태로 반환합니다. (0번째가 weight, 1번째가 bias)
weight, bias = model.layers[-1].get_weights()

# 입력 특성이 1개라 값이 하나씩만 있으므로 인덱스로 숫자 하나만 꺼냅니다.
#   - weight 는 (입력 특성 수, 출력 수) = (1, 1) 형태의 2차원 배열이라, 숫자 하나를 꺼내려면 [0][0] 로 두 번 인덱싱합니다.
#   - bias   는 (출력 수,) = (1,) 형태의 1차원 배열이라, [0] 로 한 번만 인덱싱합니다.
a = weight[0][0]    # PyTorch의 model.linear.weight 에 대응 (우리 기호로는 a)
b = bias[0]         # PyTorch의 model.linear.bias 에 대응   (우리 기호로는 b)

print("학습된 a(weight):", a)
print("학습된 b(bias):", b)
print("\nweight 배열 원본:\n", weight)
print("bias 배열 원본:\n", bias)

학습된 a(weight): 5.413974
학습된 b(bias): 1.9975419e-08

weight 배열 원본:
 [[5.413974]]
bias 배열 원본:
 [1.9975419e-08]


In [15]:
# 학습 후에도 model(X_norm)의 출력은 H(선형 계산값)입니다. (확률 아님)
H = model(X_norm)

# 예측 확률(probability = z)을 보려면 H에 sigmoid를 직접 적용합니다.
probability = tf.sigmoid(H)

print("학습 후 H 예시:")
print(H.numpy()[:5])

print("\n학습 후 probability(=sigmoid(H), 즉 z) 예시:")
print(probability.numpy()[:5])

print("\n실제 정답 y:")
print(y[:5])
# y가 0인 데이터는 probability가 0에 가깝고, y가 1인 데이터는 1에 가까우면 학습이 잘 된 것입니다.

학습 후 H 예시:
[[-7.2636085]
 [-2.4212027]
 [ 2.4212027]
 [ 7.2636085]]

학습 후 probability(=sigmoid(H), 즉 z) 예시:
[[7.000850e-04]
 [8.157011e-02]
 [9.184299e-01]
 [9.992999e-01]]

실제 정답 y:
[[0.]
 [0.]
 [1.]
 [1.]]


In [16]:
# 키가 185cm인 사람이 농구선수인지 예측합니다. (이전 PyTorch 노트북과 같은 입력값)
input_value = 185

# 새 입력값도 학습 때와 '같은 기준'으로 정규화합니다.
# 학습 때 사용한 X_mean, X_std 를 그대로 다시 씁니다. (새로 계산하면 안 됩니다.)
input_norm = (input_value - X_mean) / X_std

# Dense(1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 하므로 [[...]] 로 감쌉니다.
input_norm_array = np.array([[input_norm]], dtype=np.float32)

# model의 출력은 H_new 입니다. (선형 계산값 - 확률 아님)
H_new = model(input_norm_array)

# 예측 확률을 보려면 H_new에 sigmoid를 직접 적용합니다. (probability = sigmoid(H_new) = z)
probability = tf.sigmoid(H_new)

# [prediction = 1 if ... else 0 - 한 줄 조건문]
# 이 코드는 if/else 조건문을 한 줄로 짧게 쓴 형태입니다.
#   - probability가 0.5 이상이면 prediction에 1을 넣고,
#   - 그렇지 않으면 prediction에 0을 넣습니다.
#
# [probability.numpy().item() 은 왜 쓰나요?]
# H_new와 probability는 shape이 (1, 1)인 값입니다. (숫자 하나가 표 안에 들어 있는 형태)
#   - .numpy() : Tensor를 NumPy 배열로 바꿉니다.
#   - .item()  : 그 배열 안에 들어 있는 '숫자 하나'만 꺼냅니다.
# 비교와 출력에서 [[...]] 같은 배열 모양이 아니라 깔끔한 숫자 하나로 다루기 위한 것입니다.
prediction = 1 if probability.numpy().item() >= 0.5 else 0

print("새 입력값:", input_value)
print("정규화된 입력값:", input_norm)
print("H_new:", H_new.numpy().item())
print("probability:", probability.numpy().item())
print("prediction:", prediction)

print(f"\n키가 {input_value}cm인 사람의 농구선수 확률(probability): {probability.numpy().item():.4f}")
if prediction == 1:
    print("판별 결과: 농구선수입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")

새 입력값: 185
정규화된 입력값: 0.8944271969412381
H_new: 4.842405319213867
probability: 0.9921736717224121
prediction: 1

키가 185cm인 사람의 농구선수 확률(probability): 0.9922
판별 결과: 농구선수입니다.
